# Pronunciation Assessment — record and assess in the browser

A small web interface for the same real-time endpoint the first notebook deploys: pick a
language, type what the speaker should say, **record yourself**, and read the assessment
as coloured words, a per-word table and per-phoneme detail — rather than as printed JSON.

---

## Before you start

1. **Deploy the endpoint first.** Run
   [`assess-pronunciation-realtime.ipynb`](assess-pronunciation-realtime.ipynb) sections
   1 to 3 — subscribe on AWS Marketplace, then deploy. This notebook only *calls* an
   endpoint; it never creates one, and it never deletes one.
2. Make sure `ENDPOINT_NAME` in section 2 below matches the one that notebook created.
3. Run this notebook with an IAM role that may call `sagemaker:DescribeEndpoint` and
   `sagemaker:InvokeEndpoint`.

## How you will reach the app

Browsers only grant microphone access in a **secure context** — HTTPS, or `localhost`.
The little frame Gradio can render under a cell is neither, so recording does not work
there.

What does work: SageMaker already serves your notebook over HTTPS on an AWS domain, and
its Jupyter server proxies local ports. Section 8 starts the app on a local port and
prints the proxied `https://…sagemaker.aws/…` URL — **open that in a new browser tab**
and the microphone works. Nothing is published outside your account, and nobody who is
not signed in to this SageMaker domain can reach it.

## Cost

Real-time software charges are per request, so an endpoint nobody is talking to costs
you only its instance — but that instance bills by the hour for as long as the endpoint
exists. **This notebook does not delete it.** When you are done, run
section 9 of [`assess-pronunciation-realtime.ipynb`](assess-pronunciation-realtime.ipynb).


---
## 1. Install dependencies

`gradio` is the interface, `requests-toolbelt` builds the multipart body and reports the
exact `Content-Type` that matches it.

The SageMaker Python SDK is deliberately *not* here: invoking an endpoint needs nothing
beyond `boto3`, and leaving the SDK out keeps it from arguing with Gradio over `pydantic`
and `fastapi` versions.

Gradio pulls in a fair number of packages. If the imports in section 3 fail straight
after this, restart the kernel and run again.

In [ ]:
%pip install -q "boto3>=1.34" "requests-toolbelt>=1.0" "gradio>=5,<6"

---
## 2. Configure

`ENDPOINT_NAME` must match the endpoint deployed by the first notebook — it is that
notebook's `ENDPOINT_NAME`, unchanged.

`APP_PORT` is the local port the interface listens on. Change it only if something else
on this instance already holds it; section 8 builds the proxy URL from whatever you set
here.

In [ ]:
ENDPOINT_NAME = "ekinox-pronunciation-assessment-demo"
APP_PORT = 7860

---
## 3. Check the endpoint is up

A clear failure here beats a confusing one from inside the interface later.

In [ ]:
import boto3
from botocore.exceptions import ClientError

boto_session = boto3.Session()
region = boto_session.region_name
sagemaker_client = boto_session.client("sagemaker")
runtime = boto_session.client("sagemaker-runtime")

try:
    endpoint = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
except ClientError as error:
    # "Could not find endpoint" is a ValidationException, not a 404.
    if error.response["Error"]["Code"] != "ValidationException":
        raise
    raise RuntimeError(
        f"No endpoint named {ENDPOINT_NAME} in {region}.\n"
        f"Deploy one by running sections 1 to 3 of "
        f"assess-pronunciation-realtime.ipynb, and check that ENDPOINT_NAME "
        f"above matches the one it creates."
    ) from None

status = endpoint["EndpointStatus"]
if status != "InService":
    raise RuntimeError(
        f"Endpoint {ENDPOINT_NAME} is {status}, not InService. "
        f"{endpoint.get('FailureReason', 'Wait for it to come up, or redeploy it.')}"
    )

config = sagemaker_client.describe_endpoint_config(
    EndpointConfigName=endpoint["EndpointConfigName"])

print(f"Region:        {region}")
print(f"Endpoint:      {ENDPOINT_NAME}  ({status})")
print(f"Instance type: {config['ProductionVariants'][0]['InstanceType']}")

---
## 4. Talk to the endpoint

The same two functions as
[`assess-pronunciation-realtime.ipynb`](assess-pronunciation-realtime.ipynb) sections 4
and 5, repeated verbatim so this notebook stands on its own.

`MultipartEncoder.content_type` is the piece that matters: it is the
`multipart/form-data; boundary=...` string describing the body the encoder just built.
Pass that same string as `ContentType` and the two always agree.

In [ ]:
import json
import os
import time

from requests_toolbelt.multipart.encoder import MultipartEncoder


def build_request(audio_path, expected_text, language="fr-fr"):
    """Return (body_bytes, content_type) for one assessment request."""
    with open(audio_path, "rb") as audio_file:
        encoder = MultipartEncoder(
            fields={
                "language": language,
                "expectedText": expected_text,
                # The filename and content type are ignored by the model, but
                # most HTTP libraries will not emit a file part without them.
                "audio": (os.path.basename(audio_path), audio_file, "audio/wav"),
            }
        )
        # Read the body before the file closes.
        return encoder.to_string(), encoder.content_type


def assess(audio_path, expected_text, language="fr-fr", attempts=12):
    """Invoke the endpoint, retrying while the model is still warming up."""
    body, content_type = build_request(audio_path, expected_text, language)

    for attempt in range(1, attempts + 1):
        try:
            response = runtime.invoke_endpoint(
                EndpointName=ENDPOINT_NAME,
                ContentType=content_type,
                Accept="application/json",
                Body=body,
            )
            return json.loads(response["Body"].read())
        except ClientError as error:
            # 503 means the model has not finished loading.
            status = error.response.get("OriginalStatusCode")
            if status != 503 or attempt == attempts:
                raise
            # Cap the backoff: doubling indefinitely would spend the
            # whole budget asleep. 12 attempts caps out near two minutes.
            wait = min(2 ** attempt, 15)
            print(f"Model still warming up, retrying in {wait}s...")
            time.sleep(wait)

---
## 5. Turn a recording into a WAV the model accepts

The model reads **16-bit integer PCM in a RIFF/WAVE container, and nothing else** —
anything more exotic comes back as a `400`. A browser recording arrives here as a plain
array of samples, which may be float or integer depending on the browser, so write the
WAV ourselves: it is a dozen lines of standard library, and it means this notebook needs
no `ffmpeg` on the instance.

Sample rate and channel count are left exactly as recorded. The model resamples to 16 kHz
and downmixes to mono internally, so there is nothing to gain by doing it here.

In [ ]:
import tempfile
import wave

import numpy as np


def peak_level(recording):
    """Loudest sample of a recording, as a fraction of full scale (0.0 to 1.0).

    Normalised across dtypes so one threshold covers whatever the browser
    hands over: float PCM already runs -1.0 to 1.0, integer PCM is divided by
    its own full scale.
    """
    samples = recording[1]
    if samples.size == 0:
        return 0.0

    if samples.dtype.kind == "f":
        return min(float(np.abs(samples).max()), 1.0)

    info = np.iinfo(samples.dtype)
    if samples.dtype.kind == "u":
        # Unsigned PCM is centred on its midpoint rather than on zero, and its
        # `min` is 0 — dividing by that would raise. Measure the excursion
        # either side of the midpoint instead.
        midpoint = (float(info.max) + 1.0) / 2.0
        peak = float(np.abs(samples.astype(np.float64) - midpoint).max())
        return min(peak / midpoint, 1.0)

    peak = float(np.abs(samples.astype(np.float64)).max())
    return peak / abs(float(info.min))


def write_pcm16_wav(recording, path):
    """Write Gradio's (sample_rate, samples) tuple as a PCM16 WAV file."""
    sample_rate, samples = recording

    if samples.dtype.kind == "f":
        # Float PCM runs -1.0 to 1.0. Clip before scaling: a sample fractionally
        # outside the range would wrap around to the opposite extreme and click.
        samples = (np.clip(samples, -1.0, 1.0) * 32767).astype(np.int16)
    elif samples.dtype == np.int32:
        # 32-bit integer PCM has to be rescaled, not truncated — a plain cast
        # keeps the low bits, which is noise rather than audio.
        samples = (samples >> 16).astype(np.int16)
    elif samples.dtype == np.uint8:
        # 8-bit WAV is unsigned and centred on 128. Recentre, then scale up to
        # 16-bit range: a plain cast would leave a near-silent recording sitting
        # on a large DC offset.
        samples = ((samples.astype(np.int16) - 128) << 8).astype(np.int16)
    elif samples.dtype == np.int8:
        # Signed 8-bit needs the same scaling, without the recentring.
        samples = (samples.astype(np.int16) << 8).astype(np.int16)
    elif samples.dtype != np.int16:
        samples = samples.astype(np.int16)

    # Mono arrives one-dimensional; multi-channel as (frames, channels), which
    # is already the interleaved layout a WAV file wants.
    channels = 1 if samples.ndim == 1 else samples.shape[1]

    with wave.open(path, "wb") as wav_file:
        wav_file.setnchannels(channels)
        wav_file.setsampwidth(2)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(np.ascontiguousarray(samples).tobytes())

    return path

---
## 6. Render the result

A handful of small functions turning one response into HTML. They are plain Python — nothing here
knows about Gradio — so you can lift them straight into your own UI.

The interesting one is the sentence view, which highlights the letters the speaker got
wrong using `words[].mispronouncedGraphemes`. Three things about those spans, all from
[`../docs/api.md`](../docs/api.md#grapheme-spans):

* offsets index **that word's own `referenceWord`**, not the sentence you submitted;
* a `mispronounced` word can still carry *no* spans, when the fault maps to no letters —
  so fall back to styling the whole word;
* only `mispronounced` words carry spans at all: an `omitted` word has none, even though
  none of it was said.

`scorePercent` is a true 0–100 percentage at every level, and higher is better — but it is
a confidence readout, not a pass mark. Whether the attempt passed is `accepted`, decided
for you. The banner below shows both, and says which is which.

In [ ]:
import html

# Foreground and background per word status. Both are set explicitly so the
# chips read the same in a light or a dark browser theme.
STATUS_COLOURS = {
    "correct": ("#14532d", "#dcfce7"),
    "mispronounced": ("#7c2d12", "#ffedd5"),
    "omitted": ("#3f3f46", "#e4e4e7"),
    "insertion": ("#4c1d95", "#ede9fe"),
}

LEGEND = "".join(
    '<span style="display:inline-block;margin-right:.6rem;padding:.1rem .45rem;'
    f'border-radius:.3rem;font-size:.8rem;background:{background};color:{colour}">'
    f'{status}</span>'
    for status, (colour, background) in STATUS_COLOURS.items()
)


def underline_spans(word, spans):
    """Return `word` as HTML, with every character range in `spans` underlined.

    Spans are half-open and index `word` itself. Unlike the per-phoneme
    `graphemes`, a word's `mispronouncedGraphemes` do not overlap — the
    `cursor` check below is belt and braces, so a surprise can never produce
    crossed tags.
    """
    pieces = []
    cursor = 0
    for span in sorted(spans, key=lambda span: span["start"]):
        start, end = span["start"], span["end"]
        if start < cursor:
            continue
        pieces.append(html.escape(word[cursor:start]))
        pieces.append(
            # color:inherit is inline, so it outranks any theme rule targeting
            # spans and keeps the chip's own colour. See verdict_html().
            '<span style="text-decoration:underline wavy;text-underline-offset:.25em;'
            'color:inherit">'
            + html.escape(word[start:end])
            + "</span>"
        )
        cursor = end
    pieces.append(html.escape(word[cursor:]))
    return "".join(pieces)


def verdict_html(result, level=None):
    """The headline: did it pass, and how close was it.

    `level` is the recording's peak as a fraction of full scale, shown so a
    poor score that is really a quiet microphone can be recognised as one.
    """
    colour, background = ("#14532d", "#dcfce7") if result["accepted"] else ("#7c2d12", "#ffedd5")
    verdict = "ACCEPTED" if result["accepted"] else "NOT ACCEPTED"
    faults = ", ".join(result["faults"]) or "none"

    # The colour is repeated on every element holding text, rather than set once
    # on the wrapper: a dark theme sets `color` on inner elements directly, and a
    # direct rule beats inheritance — dark text would otherwise turn light while
    # the light background stayed, leaving the banner unreadable.
    return (
        f'<div style="padding:1rem 1.2rem;border-radius:.6rem;background:{background};color:{colour}">'
        f'<div style="font-size:1.5rem;font-weight:700;letter-spacing:.02em;color:{colour}">{verdict}</div>'
        f'<div style="font-size:1.05rem;margin-top:.2rem;color:{colour}">score {result["scorePercent"]:.1f}%'
        '<span style="opacity:.7;color:inherit"> — how close, not whether it passed</span></div>'
        f'<div style="font-size:.85rem;opacity:.75;margin-top:.35rem;color:{colour}">'
        f'language {result["language"]} · faults: {html.escape(faults)} · '
        f'model {html.escape(result["assessmentVersion"])}'
        + ("" if level is None else f' · recording level {level:.0%}')
        + "</div></div>"
    )


def sentence_html(result):
    """One chip per word, coloured by status, mispronounced letters underlined."""
    chips = []
    for word in result["words"]:
        colour, background = STATUS_COLOURS[word["status"]]
        extra = ""

        if word["status"] == "insertion":
            # No reference word to show — this is something the speaker added.
            label = "+" + html.escape(word["producedText"] or "?")
        elif word["mispronouncedGraphemes"]:
            label = underline_spans(word["referenceWord"], word["mispronouncedGraphemes"])
        else:
            label = html.escape(word["referenceWord"] or "?")
            if word["status"] == "mispronounced":
                # Mispronounced, but the fault maps to no particular letters.
                extra = "text-decoration:underline wavy;text-underline-offset:.25em;"
            elif word["status"] == "omitted":
                extra = "text-decoration:line-through;opacity:.7;"

        chips.append(
            '<span style="display:inline-block;margin:0 .3rem .45rem 0;padding:.35rem .65rem;'
            f'border-radius:.45rem;font-size:1.35rem;background:{background};color:{colour};'
            f'{extra}">{label}</span>'
        )

    # Arabic reads right to left, so the chips have to as well.
    direction = "rtl" if result["language"] == "ar" else "ltr"
    return (
        f'<div dir="{direction}" style="line-height:2.4;margin:.6rem 0">{"".join(chips)}</div>'
        f'<div dir="ltr" style="margin-top:.4rem">{LEGEND}</div>'
    )


def word_rows(result):
    """Rows for the per-word table: expected, status, score, what was said."""
    return [
        [
            word["referenceWord"] or "—",
            word["status"],
            "—" if word["scorePercent"] is None else f"{word['scorePercent']:.1f}%",
            word["producedText"] or "—",
        ]
        for word in result["words"]
    ]


def phonemes_html(result):
    """Per-phoneme detail for every word, in IPA.

    `tolerated` is worth reading alongside `status`: a tolerated phoneme keeps
    status `correct` while scoring low — the speaker said something else and a
    rule forgave it.
    """
    blocks = []
    for word in result["words"]:
        rows = "".join(
            "<tr>"
            f'<td style="padding:.15rem .7rem .15rem 0">{html.escape(phoneme["referencePhoneme"] or "—")}</td>'
            f'<td style="padding:.15rem .7rem .15rem 0">{html.escape(phoneme["producedPhoneme"] or "—")}</td>'
            f'<td style="padding:.15rem .7rem .15rem 0;text-align:right">{phoneme["scorePercent"]:.1f}%</td>'
            f'<td style="padding:.15rem .7rem .15rem 0">{phoneme["status"]}</td>'
            f'<td style="padding:.15rem 0;opacity:.7">{"tolerated" if phoneme["tolerated"] else ""}</td>'
            "</tr>"
            for phoneme in word["phonemes"]
        )
        blocks.append(
            f'<div style="margin-bottom:1rem"><div style="font-weight:600">'
            f'{html.escape(word["referenceWord"] or "—")} '
            f'<span style="font-weight:400;opacity:.7">({word["status"]})</span></div>'
            '<table style="border-collapse:collapse;font-size:.9rem;margin-top:.2rem">'
            '<tr style="opacity:.6;text-align:left">'
            "<th>expected</th><th>produced</th><th>score</th><th>status</th><th></th></tr>"
            f"{rows}</table></div>"
        )
    return "".join(blocks)

---
## 7. Build the interface

Left: what you are assessing. Right: the result, with the detail tucked into accordions
so the verdict stays readable.

Failures surface as a message in the interface rather than a traceback in the notebook.
The mapping is the model's, from
[`../docs/api.md`](../docs/api.md):

| `OriginalStatusCode` | Meaning |
|---|---|
| `400` | The request is multipart but unusable — a missing part, blank `expectedText`, an unsupported language, or audio that is not decodable PCM16 WAV |
| `415` | The `ContentType` is not `multipart/form-data` at all |
| `503` | The model is still loading — `assess()` already retries this one |
| `500` | An unexpected internal failure |

A recording in which every sample is zero is refused before it is sent. The endpoint would
happily score it — as a total omission, 0%, every word missing — and that reads like a
pronunciation verdict rather than what it is: a microphone that was never really open.

The progress bar covers this notebook's own work, encoding and the round trip. It cannot
cover the gap between pressing stop and the audio arriving: that happens in the browser,
before anything here runs.

Nothing runs at build time: `cache_examples=False` keeps the example from being assessed
the moment the app starts, which would bill you an invocation before anyone has clicked
anything.

In [ ]:
import gradio as gr

LANGUAGES = {"French": "fr-fr", "Arabic": "ar-sa"}
SAMPLE_AUDIO = "../data/sample_audio.wav"

# Written for this notebook, so there is nothing to license, and chosen to
# exercise what the highlighting can show rather than to score well. Each
# carries the language's own "letters you do not say as written" case — the
# thing a learner misses and `mispronouncedGraphemes` points at:
#
#   French  two obligatory liaisons (les‿enfants, enfants‿ont), a droppable
#           schwa in "petite", and the /y/ of "une".
#   Arabic  two sun-letter assimilations (at-tuffāḥ, aṣ-ṣaghīra, where the
#           lām of the article is written but not pronounced), the emphatics
#           ط and ص, and ح and غ.
#
# Left unvocalised, as an application would send it. If your speakers need the
# diacritics to read it aloud, add them — and check the grapheme offsets still
# land where you expect, since they count characters.
REFERENCE_TEXT = {
    "French": "Les enfants ont mangé une petite tarte aux pommes.",
    "Arabic": "الأطفال أكلوا فطيرة التفاح الصغيرة.",
}

ERROR_HELP = {
    400: "The model rejected the request. Most likely the audio is not 16-bit PCM WAV, "
         "the expected text is blank, or the language is unsupported.",
    415: "The request did not arrive as multipart/form-data.",
    503: "The model is still warming up — give it a few seconds and try again.",
    500: "The model failed unexpectedly. Check the endpoint's CloudWatch logs.",
}


def run_assessment(recording, expected_text, language_label, progress=gr.Progress()):
    """Gradio callback: one recording in, five rendered outputs out.

    Gradio fills `progress` in itself — declaring it as a default argument is
    how you ask for a progress bar, and it is not one of the inputs below. It
    covers this function only: encoding, then the round trip to the endpoint.
    What the recorder does between pressing stop and handing the audio over
    happens in the browser, before any of this runs, and cannot be reported on
    from here.
    """
    if recording is None:
        raise gr.Error("Record something, or upload a WAV file, first.")
    if not expected_text or not expected_text.strip():
        raise gr.Error("Type the text the speaker was supposed to say.")

    # Normal speech peaks somewhere between 10% and 80% of full scale. Below a
    # fiftieth of it the endpoint reports every word omitted — a 0% that reads
    # like a verdict on the speaker rather than on the microphone. The bar is
    # set well under any usable recording so that only a broken input trips it.
    level = peak_level(recording)
    if level < 0.02:
        raise gr.Error(
            f"That recording peaks at {level:.2%} of full scale — far too quiet to "
            f"assess. Check which input device the browser chose, and the input "
            f"volume for it. Note that the recorder's own waveform is scaled to fit, "
            f"so a near-silent recording still looks healthy while it is being made."
        )

    progress(0.1, desc="Encoding as 16-bit PCM WAV")

    # NamedTemporaryFile(delete=False) then unlink: the file has to still be
    # there when build_request() reopens it by name.
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as temporary:
        audio_path = temporary.name
    try:
        write_pcm16_wav(recording, audio_path)
        progress(0.4, desc="Assessing on the endpoint")
        result = assess(audio_path, expected_text.strip(), LANGUAGES[language_label])
        progress(0.9, desc="Rendering")
    except ClientError as error:
        status = error.response.get("OriginalStatusCode")
        detail = error.response.get("OriginalMessage") or str(error)
        raise gr.Error(f"{ERROR_HELP.get(status, 'The request failed.')} ({status}: {detail})")
    finally:
        os.unlink(audio_path)

    return (
        verdict_html(result, level),
        sentence_html(result),
        word_rows(result),
        phonemes_html(result),
        json.dumps(result, indent=2, ensure_ascii=False),
    )


with gr.Blocks(title="Pronunciation Assessment", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Pronunciation Assessment\n"
        "Pick a language, type what should be said, record yourself, and press "
        "**Assess**. Everything stays inside your AWS account."
    )

    with gr.Row():
        with gr.Column(scale=2):
            language_input = gr.Dropdown(
                choices=list(LANGUAGES), value="French", label="Language")
            text_input = gr.Textbox(
                label="Expected text",
                value=REFERENCE_TEXT["French"],
                placeholder="What the speaker is supposed to say",
            )
            audio_input = gr.Audio(
                sources=["microphone", "upload"],
                type="numpy",
                label="Recording",
                # The trim editor runs a WebAssembly ffmpeg build in the browser,
                # which is a lot of machinery to load through a proxy for a feature
                # this app does not use. Assess the whole recording.
                editable=False,
            )
            assess_button = gr.Button("Assess", variant="primary")

            if os.path.exists(SAMPLE_AUDIO):
                gr.Examples(
                    examples=[[SAMPLE_AUDIO, REFERENCE_TEXT["French"], "French"]],
                    inputs=[audio_input, text_input, language_input],
                    label="Sample recording (French, deliberately imperfect)",
                    cache_examples=False,
                )

        with gr.Column(scale=3):
            verdict_output = gr.HTML()
            sentence_output = gr.HTML()
            word_output = gr.Dataframe(
                headers=["expected", "status", "score", "produced"],
                datatype=["str", "str", "str", "str"],
                label="Words",
                wrap=True,
            )
            with gr.Accordion("Phoneme detail (IPA)", open=False):
                phoneme_output = gr.HTML()
            with gr.Accordion("Raw JSON response", open=False):
                json_output = gr.Code(language="json")

    def load_reference_text(language_label):
        """Swap in that language's reference sentence, and its writing direction.

        Changing language invalidates whatever is in the box — the text and the
        recording have to be in the same language for the score to mean
        anything — so replacing it is the helpful move rather than a
        destructive one.
        """
        return gr.update(
            value=REFERENCE_TEXT[language_label],
            rtl=(language_label == "Arabic"),
        )

    language_input.change(
        load_reference_text, inputs=language_input, outputs=text_input)

    assess_button.click(
        run_assessment,
        inputs=[audio_input, text_input, language_input],
        outputs=[verdict_output, sentence_output, word_output, phoneme_output, json_output],
        show_progress="full",
    )

print("Interface built. Section 8 starts it.")

---
## 8. Start the app, and open it in a new tab

The app listens on `127.0.0.1:APP_PORT` — loopback only, so the sole way in is the Jupyter
server already running in front of you, which proxies local ports over its own HTTPS
origin. Two flavours of notebook, two prefixes:

| Environment | URL |
|---|---|
| Notebook instance | `https://<name>.notebook.<region>.sagemaker.aws/proxy/<port>/` |
| Studio JupyterLab | `https://<domain-id>.studio.<region>.sagemaker.aws/jupyterlab/default/proxy/<port>/` |

`root_path` tells Gradio it is being served under that prefix, so the asset and API URLs
it emits point back through the proxy instead of at the root.

**Open the printed URL in a new browser tab** — not the frame under the cell, which is not
a secure context and gets no microphone. Allow microphone access when the browser asks.

If the interface reports that your recording is too quiet, the cause is upstream of this
notebook: the browser picked an input, and it is not the one you are speaking into, or its
level is far too low. Check the browser's own microphone setting for this site as well as
the operating system's — they are separate, and they disagree more often than you would
think. Trying a second browser is the quickest way to tell a device problem from a browser
one: Firefox, Chrome and Safari have all been used successfully, and at least one
privacy-hardened browser has been seen to record at a hundredth of the level the same
machine produces elsewhere.

If the URL does not resolve, the prefix is the thing to adjust, and you can derive it by
hand: take the address bar of this notebook, cut everything from `/lab` onwards, and
append `proxy/<port>/`. If the page loads but is blank, try `/proxy/absolute/<port>/`
instead — that variant does not strip the prefix before forwarding — and pass the same
path as `root_path`.

In [ ]:
RESOURCE_METADATA = "/opt/ml/metadata/resource-metadata.json"


def proxy_location(port, region):
    """Return (root_path, url) for reaching a local port through SageMaker.

    Both flavours run behind a Jupyter server that proxies local ports, but
    they mount it under different prefixes and sit on different domains. The
    metadata file is what tells them apart: a notebook instance reports a
    ResourceName, Studio reports a DomainId.

    Returns (None, None) outside SageMaker, or if the file is not what we
    expect — better to say so than to print a URL that cannot work.
    """
    try:
        with open(RESOURCE_METADATA) as metadata_file:
            metadata = json.load(metadata_file)
    except (OSError, ValueError):
        return None, None

    if "DomainId" in metadata:
        # "default" is the app name Studio uses for the JupyterLab app it opens.
        root_path = f"/jupyterlab/default/proxy/{port}"
        host = f"{metadata['DomainId']}.studio.{region}.sagemaker.aws"
    elif "ResourceName" in metadata:
        root_path = f"/proxy/{port}"
        host = f"{metadata['ResourceName']}.notebook.{region}.sagemaker.aws"
    else:
        return None, None

    return root_path, f"https://{host}{root_path}/"


root_path, app_url = proxy_location(APP_PORT, region)

demo.launch(
    server_port=APP_PORT,
    share=False,             # Gradio turns this on by itself on SageMaker; we do not
                             # want a public tunnel, the proxy URL below is the way in
    root_path=root_path,
    inline=False,            # the frame under the cell gets no microphone
    prevent_thread_lock=True,  # keep the kernel free, so section 9 can close the app
    quiet=True,
)

print(f"Running on 127.0.0.1:{APP_PORT}.\n")
if app_url:
    print("Open this in a NEW BROWSER TAB, and allow microphone access:\n")
    print(f"    {app_url}\n")
else:
    print("Could not work out this environment's proxy URL. Build it by hand:\n")
    print("  take the address bar of this notebook, cut everything from /lab")
    print(f"  onwards, and append  proxy/{APP_PORT}/\n")
print("Leave this kernel running — the app dies with it.")

---
## 9. Stop the app

The app runs until you stop it or the kernel dies. Closing it frees the port, so you can
re-run section 8 after editing the interface.

In [ ]:
demo.close()

print("App stopped.\n")
print(f"The endpoint {ENDPOINT_NAME} is still running — its instance bills by the hour.")
print("Run section 9 of assess-pronunciation-realtime.ipynb to delete it.")

---
## Where to go next

- [`../docs/api.md`](../docs/api.md) — every request and response field, plus
  [Batch Transform](../docs/api.md#batch-transform) and its shared boundary
- [`../docs/openapi.yaml`](../docs/openapi.yaml) — the same contract as OpenAPI 3.0
- [`assess-pronunciation-realtime.ipynb`](assess-pronunciation-realtime.ipynb) — deploy,
  invoke and delete the endpoint; also covers the error cases
- [`../README.md`](../README.md) — supported languages, instance types, versioning

The interface built in section 7 is an ordinary `gr.Blocks`, and the renderers in section
6 are ordinary functions: both lift out of this notebook unchanged. Run them as a script
on your own machine — `localhost` is a secure context too, so the microphone works there
without a proxy — or put them in a container behind an authenticating front end when you
want something that outlives a notebook kernel.

Questions about the product go through the support channel on the AWS Marketplace
listing. Problems with this notebook:
[open an issue](https://github.com/EkinoxIO/ekinox-marketplace/issues).